In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
import pickle


In [4]:
# 1. Load the cleaned dataset
df = pd.read_csv('../data/clean_orders/orders.csv')

# 2. Convert dates to datetime to calculate 'expected_days'
df['order_date'] = pd.to_datetime(df['order_date'])
df['expected_delivery_date'] = pd.to_datetime(df['expected_delivery_date'])
df['expected_days'] = (df['expected_delivery_date'] - df['order_date']).dt.days

# 3. NO LEAKAGE RULE: Select ONLY the allowed features + target
# We specifically do NOT include actual_delivery_days, delivery_status, delivery_delay, etc.
allowed_columns = [
    'courier', 'expected_days', 'weather', 'season', 
    'area', 'category', 'order_month', 'order_hour', 
    'is_late' # Target
]

# Create our strict modeling dataframe and drop any rows with NaN in these specific columns
model_df = df[allowed_columns].dropna().copy()

print(f"Dataset shape ready for ML: {model_df.shape}")


Dataset shape ready for ML: (113314, 9)


In [5]:
# 1. Define X (Features) and y (Target)
X = model_df.drop(columns=['is_late'])
y = model_df['is_late']

# 2. Train/Test Split (80/20) - Stratified ensures same % of late orders in both sets!
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Identify categorical vs numerical columns
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

# 4. Preprocessing: Scale numbers, One-Hot Encode categories
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

# Fit on training data, transform both
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

# Get the new column names after OneHotEncoding
encoded_cat_cols = preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols)
all_feature_names = num_cols + list(encoded_cat_cols)

print(f"Number of features after encoding: {X_train_encoded.shape[1]}")


Number of features after encoding: 114


In [6]:
# Save the exact arrays to a pickle file so Tasks 2 & 3 can load them
export_data = {
    'X_train': X_train_encoded,
    'X_test': X_test_encoded,
    'y_train': y_train,
    'y_test': y_test,
    'feature_names': all_feature_names
}

# Save it to the data folder
with open('../data/model2_ready_data.pkl', 'wb') as f:
    pickle.dump(export_data, f)
    
